# GeoLife CP1 — Cleaning + Stay-point Validation

**Mục tiêu:** validate implementation production của cleaning + stay-point detector trên synthetic acceptance cases và GeoLife thật.

Notebook này **không chứa implementation thứ hai**. Nó import trực tiếp từ `src/geolife/` để notebook và production dùng cùng source of truth.


## Cách chạy

Thiết kế theo workflow notebook EDA hiện tại:

- ưu tiên Modal volume tại `/mnt/geolife-data`;
- hỗ trợ `GEOLIFE_DATA_ROOT` và `GEOLIFE_REPO_DIR` khi chạy nơi khác;
- clone/pull branch `cp1-cleaning-staypoint` nếu chưa chạy từ repo local;
- cache output đắt tiền dưới volume;
- có thể chạy `Run All` từ đầu đến cuối.

Baseline ban đầu:

- same-second radius: **10 m**
- max gap: **300 s**
- hard-speed guard: **1200 km/h**
- stay distance: **200 m**
- min dwell: **1200 s (20 phút)**


In [ ]:
from pathlib import Path
from time import perf_counter
from IPython.display import display, Markdown
import os
import subprocess
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp1-cleaning-staypoint")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
CACHE_DIR = Path(os.environ.get("GEOLIFE_CACHE_DIR", "/mnt/geolife-data/cache/cp1_staypoints"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root() -> Path:
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data"),
        Path("/mnt/geolife-data/Data"),
        Path("/mnt/geolife-data/extracted/Data"),
    ])

    checked = []
    for candidate in candidates:
        checked.append(str(candidate))
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate

    volume_root = Path("/mnt/geolife-data")
    if volume_root.is_dir():
        for candidate in sorted(volume_root.glob("**/Data")):
            if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
                return candidate

    raise FileNotFoundError(
        "Không tìm thấy GeoLife Data folder có */Trajectory/*.plt. "
        f"Đã kiểm tra: {checked}. Set GEOLIFE_DATA_ROOT nếu dataset nằm nơi khác."
    )

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from geolife.staypoints import clean_trajectory, detect_staypoints

print("Repo:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Cache:", CACHE_DIR)


# 1. Contract + baseline configuration

Notebook chỉ kiểm chứng implementation đã được contract hóa; tuning parameter không thay đổi algorithm semantics.


In [ ]:
BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}
BASELINE


# 2. Synthetic acceptance smoke tests

Các ví dụ dưới đây phản ánh đúng các semantics đã review trước khi chạy full dataset.


In [ ]:
def frame(rows):
    return pd.DataFrame(rows, columns=["timestamp", "latitude", "longitude"]).assign(
        timestamp=lambda x: pd.to_datetime(x["timestamp"], utc=True)
    )

# Invalid coordinate -> boundary
raw = frame([
    ("2026-01-01T10:00:00Z", 39.0, 116.0),
    ("2026-01-01T10:01:00Z", 39.0, 116.0),
    ("2026-01-01T10:02:00Z", 200.0, 116.0),
    ("2026-01-01T10:03:00Z", 39.0, 116.0),
    ("2026-01-01T10:24:00Z", 39.0, 116.0),
])
clean = clean_trajectory(raw)
display(clean)
assert clean["sequence_id"].tolist() == [0, 0, 1, 2]


Lưu ý: ví dụ trên cố tình có thêm gap 21 phút sau `C`, nên ngoài boundary do invalid coordinate còn có boundary temporal gap. Đây là check tốt để thấy các stage compose tuần tự thay vì che lấp nhau.


In [ ]:
# Compact same-second -> one representative
raw = frame([
    ("2026-01-01T10:00:00Z", 39.000000, 116.000000),
    ("2026-01-01T10:00:00Z", 39.000020, 116.000010),
    ("2026-01-01T10:00:00Z", 39.000010, 116.000020),
])
compact = clean_trajectory(raw)
display(compact)
assert len(compact) == 1
assert int(compact.iloc[0]["raw_point_count"]) == 3
assert compact.iloc[0]["max_radius_m"] <= 10.0


In [ ]:
# Terminal valid stay must be emitted
points = pd.DataFrame([
    ("2026-01-01T10:00:00Z", 39.0000, 116.0, 0),
    ("2026-01-01T10:07:00Z", 39.0002, 116.0, 0),
    ("2026-01-01T10:14:00Z", 39.0004, 116.0, 0),
    ("2026-01-01T10:21:00Z", 39.0006, 116.0, 0),
], columns=["timestamp", "latitude", "longitude", "sequence_id"])
points["timestamp"] = pd.to_datetime(points["timestamp"], utc=True)

stays = detect_staypoints(points, distance_threshold_m=200, min_dwell_s=1200)
display(stays)
assert len(stays) == 1
assert stays.iloc[0]["duration_s"] == 1260


In [ ]:
# First outside-radius point closes candidate; later return cannot bridge it.
points = pd.DataFrame([
    ("2026-01-01T10:00:00Z", 39.0000, 116.0, 0),
    ("2026-01-01T10:05:00Z", 39.0009, 116.0, 0),
    ("2026-01-01T10:10:00Z", 39.0016, 116.0, 0),
    ("2026-01-01T10:15:00Z", 39.0020, 116.0, 0),
    ("2026-01-01T10:21:00Z", 39.0004, 116.0, 0),
], columns=["timestamp", "latitude", "longitude", "sequence_id"])
points["timestamp"] = pd.to_datetime(points["timestamp"], utc=True)

assert detect_staypoints(points, distance_threshold_m=200, min_dwell_s=1200).empty
print("Synthetic smoke tests: OK")


# 3. GeoLife loader

Dùng loader tối giản cho notebook validation. Sau khi pipeline ổn định có thể chuyển loader reusable vào `src/geolife/data/`.


In [ ]:
def read_plt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        skiprows=6,
        header=None,
        names=["latitude", "longitude", "unused", "altitude", "date_days", "date", "time"],
    )
    df["timestamp"] = pd.to_datetime(df["date"] + " " + df["time"], utc=True)
    return df[["timestamp", "latitude", "longitude"]]

def trajectory_files(data_root: Path):
    return sorted(data_root.glob("*/Trajectory/*.plt"))

files = trajectory_files(DATA_ROOT)
user_dirs = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.isdigit())
print("User dirs:", len(user_dirs))
print("Trajectory files:", len(files))

if not files:
    raise RuntimeError(
        f"DATA_ROOT={DATA_ROOT} không chứa trajectory .plt. "
        "Dừng tại đây để tránh tạo cache rỗng."
    )

print("Example:", files[0])


# 4. Cleaning trên trajectory thật

Bắt đầu bằng một vài file để audit schema, sequence boundaries và diagnostic reasons trước khi full scan.


In [ ]:
sample_files = files[:5]
sample_rows = []
for path in sample_files:
    raw = read_plt(path)
    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
    )
    sample_rows.append({
        "file": str(path),
        "raw_rows": len(raw),
        "clean_rows": len(cleaned),
        "sequences": cleaned["sequence_id"].nunique() if len(cleaned) else 0,
        "boundaries": int(cleaned["boundary_before_reason"].notna().sum()) if len(cleaned) else 0,
    })

display(pd.DataFrame(sample_rows))


# 5. Stay-point detector trên vài trajectory thật

Mục tiêu ở bước này là sanity-check output, chưa phải benchmark Home/Office.


In [ ]:
stay_samples = []
for path in sample_files:
    cleaned = clean_trajectory(read_plt(path))
    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=BASELINE["distance_threshold_m"],
        min_dwell_s=BASELINE["min_dwell_s"],
    )
    if len(stays):
        x = stays.copy()
        x["file"] = str(path)
        stay_samples.append(x)

sample_stays = pd.concat(stay_samples, ignore_index=True) if stay_samples else pd.DataFrame()
display(sample_stays.head(30))
print("Sample stays:", len(sample_stays))


# 6. Full-release baseline audit

Cell này có thể tốn thời gian; lưu summary vào Modal volume để không phải scan lại mỗi lần.


In [ ]:
BASELINE_CACHE = CACHE_DIR / "baseline_summary_v2.pkl"
BASELINE_COLUMNS = [
    "file",
    "raw_rows",
    "clean_rows",
    "n_sequences",
    "n_stays",
    "invalid_coordinate_boundaries",
    "same_second_conflict_boundaries",
    "temporal_gap_boundaries",
    "hard_speed_boundaries",
]

def run_full_baseline(files):
    rows = []
    for i, path in enumerate(files, start=1):
        raw = read_plt(path)
        cleaned = clean_trajectory(
            raw,
            same_second_radius_m=BASELINE["same_second_radius_m"],
            max_gap_s=BASELINE["max_gap_s"],
            hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
        )
        stays = detect_staypoints(
            cleaned,
            distance_threshold_m=BASELINE["distance_threshold_m"],
            min_dwell_s=BASELINE["min_dwell_s"],
        )
        reasons = cleaned["boundary_before_reason"].value_counts(dropna=True).to_dict()
        rows.append({
            "file": str(path),
            "raw_rows": len(raw),
            "clean_rows": len(cleaned),
            "n_sequences": cleaned["sequence_id"].nunique() if len(cleaned) else 0,
            "n_stays": len(stays),
            "invalid_coordinate_boundaries": reasons.get("invalid_coordinate", 0),
            "same_second_conflict_boundaries": reasons.get("same_second_spatial_ambiguity", 0),
            "temporal_gap_boundaries": reasons.get("temporal_gap", 0),
            "hard_speed_boundaries": reasons.get("hard_speed_guard", 0),
        })
        if i % 500 == 0:
            print(f"{i:,}/{len(files):,}")
    return pd.DataFrame(rows, columns=BASELINE_COLUMNS)

def baseline_cache_is_valid(df: pd.DataFrame, files) -> bool:
    return (
        list(df.columns) == BASELINE_COLUMNS
        and len(df) == len(files)
        and len(df) > 0
    )

baseline_summary = None
if BASELINE_CACHE.exists():
    cached = pd.read_pickle(BASELINE_CACHE)
    if baseline_cache_is_valid(cached, files):
        baseline_summary = cached
        print("Loaded valid cache:", BASELINE_CACHE)
    else:
        print(
            "Ignoring stale/invalid cache:", BASELINE_CACHE,
            f"(rows={len(cached):,}, expected={len(files):,})"
        )

if baseline_summary is None:
    t0 = perf_counter()
    baseline_summary = run_full_baseline(files)
    if not baseline_cache_is_valid(baseline_summary, files):
        raise RuntimeError(
            f"Full baseline produced invalid summary: rows={len(baseline_summary):,}, "
            f"expected={len(files):,}. Cache was not written."
        )
    baseline_summary.to_pickle(BASELINE_CACHE)
    print(f"Saved {BASELINE_CACHE} in {(perf_counter()-t0)/60:.1f} min")

display(baseline_summary.describe(include="all"))


# 7. Baseline distributions

Không chọn threshold chỉ vì tạo ra nhiều stay hơn; mục tiêu là phát hiện pathology rõ ràng và chuẩn bị sensitivity comparison.


In [ ]:
if len(baseline_summary):
    print("Total stays:", int(baseline_summary["n_stays"].sum()))
    print("Files with >=1 stay:", int((baseline_summary["n_stays"] > 0).sum()))
    display(
        baseline_summary[
            [
                "n_sequences",
                "n_stays",
                "temporal_gap_boundaries",
                "hard_speed_boundaries",
            ]
        ].describe()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    baseline_summary["n_stays"].clip(upper=baseline_summary["n_stays"].quantile(0.99)).hist(
        bins=60, ax=ax
    )
    ax.set(title="Stay count per trajectory (clipped p99)", xlabel="stays", ylabel="trajectories")
    plt.show()


# 8. Sensitivity grid

Grid contract:

- gap: 120 / 300 / 600 s
- distance: 100 / 200 / 300 m
- dwell: 10 / 20 / 30 min

Tổng cộng 27 configs. Bắt đầu bằng subset hoặc cache theo config nếu full release quá đắt.


In [ ]:
GAPS = [120, 300, 600]
DISTANCES = [100, 200, 300]
DWELLS = [600, 1200, 1800]

grid = pd.DataFrame(
    [(g, d, w) for g in GAPS for d in DISTANCES for w in DWELLS],
    columns=["max_gap_s", "distance_threshold_m", "min_dwell_s"],
)
display(grid)
print("Configs:", len(grid))


# 9. Sensitivity runner

Mặc định chạy trên một subset để verify mechanics. Khi đã ổn, đổi `SENSITIVITY_FILES` sang full `files` hoặc chạy theo user/sample đã định trước.


In [ ]:
SENSITIVITY_FILES = files[:200]

def evaluate_config(paths, max_gap_s, distance_threshold_m, min_dwell_s):
    n_stays = 0
    files_with_stays = 0
    durations = []
    for path in paths:
        cleaned = clean_trajectory(read_plt(path), max_gap_s=max_gap_s)
        stays = detect_staypoints(
            cleaned,
            distance_threshold_m=distance_threshold_m,
            min_dwell_s=min_dwell_s,
        )
        n_stays += len(stays)
        files_with_stays += int(len(stays) > 0)
        if len(stays):
            durations.extend(stays["duration_s"].tolist())
    return {
        "n_stays": n_stays,
        "files_with_stays": files_with_stays,
        "median_duration_s": float(np.median(durations)) if durations else np.nan,
        "p90_duration_s": float(np.quantile(durations, 0.9)) if durations else np.nan,
    }

sensitivity_rows = []
for row in grid.itertuples(index=False):
    metrics = evaluate_config(
        SENSITIVITY_FILES,
        row.max_gap_s,
        row.distance_threshold_m,
        row.min_dwell_s,
    )
    sensitivity_rows.append({**row._asdict(), **metrics})

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.sort_values(["max_gap_s", "distance_threshold_m", "min_dwell_s"]))


# 10. Freeze CP1 stay-point baseline

Sau khi full-release audit + sensitivity không cho thấy pathology rõ ràng:

1. chốt baseline config;
2. ghi kết quả vào `docs/`;
3. merge production code;
4. chuyển sang Home/Office heuristic.

Nếu sensitivity đề xuất đổi `200 m / 20 min`, chỉ đổi config nếu algorithm semantics vẫn giữ nguyên.


# 11. Handoff → Home/Office

Bước tiếp theo sau stay-point:

```text
clean trajectories
    ↓
stay points
    ↓
convert UTC → Beijing local time (UTC+8) cho heuristic theo giờ
    ↓
v1: heuristic trực tiếp trên individual stays
    ↓
v2: DBSCAN gom repeated stays trước heuristic
    ↓
HOME / OFFICE candidates + heuristic confidence
```

Không dùng timezone local trong cleaning/stay detector; chỉ áp local time khi bắt đầu heuristic Home/Office.
